# LoRA safetensors → Draw Things `.ckpt` (Colab)

Converts a LoRA `.safetensors` file to the Draw Things `.ckpt` format with Draw Things' own converter
(`media-generation-kit-cli lora convert`, pinned revision), built on the free Colab CPU runtime. Use it when you
cannot run the converter locally (it has no native Windows build).

1. **Runtime → Run all** (or run the cells in order). Cell 2 builds the converter: expect **30–60 minutes** the first time.
2. Cell 3 asks you to pick your `.safetensors` file and downloads the `.ckpt`.

**Status: experimental.** The build has not been proven on Colab yet. If a cell fails, copy its last lines into the dtFlow issue.
Provenance: the pinned source revisions are printed by cell 2 and written next to the `.ckpt` as a `.json` file.
Your LoRA is processed inside your own Colab session; nothing is sent anywhere else.

**Runtime:** the free CPU runtime is enough. A T4 GPU runtime also works but gives no speed-up: the conversion and the
Swift build are CPU and disk work, and the converter is built without CUDA. Prefer the CPU runtime so the GPU quota stays free.

In [1]:
# @title 1. Install Swift toolchain and build libraries
import hashlib
import json
import os
import platform
import shlex
import shutil
import subprocess
import tempfile
import urllib.request
from pathlib import Path

SWIFT_VERSION = "6.0.3"
SWIFT_ROOT = "/opt/swift"


def sh(command, cwd=None, every=1):
    """Run a shell command and stream its output into the cell line by line."""
    print("$", command, flush=True)
    process = subprocess.Popen(
        command,
        shell=True,
        executable="/bin/bash",
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    count = 0
    for line in process.stdout:
        count += 1
        if count % every == 0:
            print(line.rstrip(), flush=True)
    code = process.wait()
    if code != 0:
        raise RuntimeError(f"command failed with exit code {code}: {command}")


def show_progress(blocks, block_size, total):
    """Print download progress about every 5 percent."""
    if total > 0:
        percent = min(100, blocks * block_size * 100 // total)
        if percent // 5 != show_progress.last:
            show_progress.last = percent // 5
            print(f"  downloaded {percent}%", flush=True)


show_progress.last = -1

if platform.system() != "Linux" or platform.machine() != "x86_64":
    raise RuntimeError("This notebook needs the x86_64 Linux Colab runtime.")

release = {}
with open("/etc/os-release") as handle:
    for line in handle:
        if "=" in line:
            key, value = line.rstrip().split("=", 1)
            release[key] = value.strip('"')
ubuntu = release.get("VERSION_ID", "")
if ubuntu not in ("22.04", "24.04"):
    raise RuntimeError(f"Unsupported Ubuntu {ubuntu!r}; this notebook supports 22.04 and 24.04.")

# Removed libdispatch-dev as it is not natively available in default Ubuntu 24.04 repos.
# We will handle dispatch.h dynamically in the compilation cell instead.
sh(
    "apt-get update && DEBIAN_FRONTEND=noninteractive apt-get install -y "
    "git clang libcurl4-openssl-dev libxml2-dev libsqlite3-dev zlib1g-dev libpng-dev libjpeg-dev "
    "libomp-dev libblas-dev liblapack-dev libopenblas-dev libssl-dev pkg-config binutils libc6-dev "
    "libstdc++-12-dev tzdata unzip"
)

folder = f"swift-{SWIFT_VERSION}-RELEASE-ubuntu{ubuntu}"
if not os.path.exists(f"{SWIFT_ROOT}/{folder}/usr/bin/swift"):
    url = (
        f"https://download.swift.org/swift-{SWIFT_VERSION}-release/ubuntu{ubuntu.replace('.', '')}/"
        f"swift-{SWIFT_VERSION}-RELEASE/{folder}.tar.gz"
    )
    os.makedirs(SWIFT_ROOT, exist_ok=True)
    archive = f"/tmp/{folder}.tar.gz"
    print("downloading", url, flush=True)
    urllib.request.urlretrieve(url, archive, show_progress)
    print("extracting (a progress line every 2000 files)", flush=True)
    sh(f"tar -xzvf {archive} -C {SWIFT_ROOT}", every=2000)
    os.remove(archive)

SWIFT_BIN = f"{SWIFT_ROOT}/{folder}/usr/bin"
if SWIFT_BIN not in os.environ["PATH"]:
    os.environ["PATH"] = SWIFT_BIN + os.pathsep + os.environ["PATH"]
sh("swift --version")

$ apt-get update && DEBIAN_FRONTEND=noninteractive apt-get install -y git clang libcurl4-openssl-dev libxml2-dev libsqlite3-dev zlib1g-dev libpng-dev libjpeg-dev libomp-dev libblas-dev liblapack-dev libopenblas-dev libssl-dev pkg-config binutils libc6-dev libstdc++-12-dev tzdata unzip
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease [1,578 B]
Get:3 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  Packages [2,060 kB]
Hit:5 http://archive.ubuntu.com/ubuntu noble InRelease
Get:6 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:7 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:9 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ Packages [79.0 kB]
Ge

In [ ]:
# @title 2. Build the converter (30–60 min the first time)
# @markdown Optional: keep the built converter in Google Drive so later sessions skip the build.
use_drive_cache = True  # @param {type:"boolean"}

REPO_URL = "https://github.com/drawthingsai/media-generation-kit.git"
PINNED_REVISION = "37e0b70092ad1a0c1d4c6b24f16e17b73f1b1fb3"
CLI_NAME = "media-generation-kit-cli"
WORK = Path("/content/mgk-build")
SOURCE = WORK / "media-generation-kit"
BINARY = SOURCE / ".build" / "release" / CLI_NAME
DRIVE_CACHE = Path("/content/drive/MyDrive/dtflow-lora-converter")
CACHED_BINARY = DRIVE_CACHE / f"{CLI_NAME}-{PINNED_REVISION[:7]}-swift{SWIFT_VERSION}-ubuntu{ubuntu}"

if use_drive_cache:
    from google.colab import drive

    drive.mount("/content/drive")

if use_drive_cache and CACHED_BINARY.is_file() and not BINARY.is_file():
    BINARY.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(CACHED_BINARY, BINARY)
    BINARY.chmod(0o755)
    print("restored the converter from Drive:", CACHED_BINARY)

if not BINARY.is_file():
    WORK.mkdir(parents=True, exist_ok=True)
    if not SOURCE.exists():
        sh(f"git clone --progress {REPO_URL} {SOURCE}")
    sh(f"git -C {SOURCE} checkout {PINNED_REVISION}")

    # Removed rm -rf .build to speed up iterative compilation
    sh("swift package resolve", cwd=SOURCE)

    # Generate a global modulemap for CZlib
    czlib_dir = Path("/tmp/CZlib")
    czlib_dir.mkdir(exist_ok=True)
    (czlib_dir / "module.modulemap").write_text(
        'module CZlib [system] {\n  header "/usr/include/zlib.h"\n  link "z"\n  export *\n}'
    )

    # Generate a global modulemap for SQLite3
    sqlite3_dir = Path("/tmp/SQLite3")
    sqlite3_dir.mkdir(exist_ok=True)
    (sqlite3_dir / "module.modulemap").write_text(
        'module SQLite3 [system] {\n  header "/usr/include/sqlite3.h"\n  link "sqlite3"\n  export *\n}'
    )

    # Linux compile fix: ZIPFoundation's in-memory FILE pointer is optional on Linux (fopencookie),
    # but the vendored source passes it straight to fseeko. Unwrap it.
    patched = 0
    for source in SOURCE.glob(
        ".build/checkouts/draw-things-community/Vendors/ZIPFoundation/Sources/ZIPFoundation/Archive+MemoryFile.swift"
    ):
        text = source.read_text()
        before = "      if append {\n        fseeko(result, 0, SEEK_END)"
        after = "      if append, let result = result {\n        fseeko(result, 0, SEEK_END)"
        if after in text:
            patched += 1
        elif text.count(before) == 1:
            source.write_text(text.replace(before, after, 1))
            patched += 1
    if patched != 1:
        raise RuntimeError("ZIPFoundation Linux patch did not apply; upstream changed, the pin needs a new patch.")
    print("ZIPFoundation Linux patch ready")

    # Patch s4nnc Store.swift to avoid missing _SQLite3Shims on Linux
    for store_swift in SOURCE.glob(".build/checkouts/s4nnc/nnc/Store.swift"):
        content = store_swift.read_text()
        if "import _SQLite3Shims" in content:
            store_swift.write_text(content.replace("import _SQLite3Shims", "import SQLite3"))
            print("Patched Store.swift SQLite3 imports")

    # Patch s4nnc DataFrameAddons.swift and Store.swift for Linux FILE pointer unwrapping
    for swift_file in SOURCE.glob(".build/checkouts/s4nnc/nnc/*.swift"):
        content = swift_file.read_text()
        new_content = content.replace("fclose(fileHandle)", "fclose(fileHandle!)")
        new_content = new_content.replace("fileno(fp)", "fileno(fp!)")
        new_content = new_content.replace("fseek(fp,", "fseek(fp!,")
        new_content = new_content.replace("ftell(fp)", "ftell(fp!)")
        new_content = new_content.replace("fclose(fp)", "fclose(fp!)")
        if new_content != content:
            swift_file.write_text(new_content)
            print(f"Patched FILE pointers in {swift_file.name}")

    # Patch libraries to avoid Apple-specific framework imports on Linux and add FoundationNetworking
    for swift_file in SOURCE.glob(".build/checkouts/draw-things-community/Libraries/**/*.swift"):
        content = swift_file.read_text()
        original_content = content

        if (
            "import Foundation" in content
            and ("URLSession" in content or "URLRequest" in content)
            and "FoundationNetworking" not in content
        ):
            content = content.replace(
                "import Foundation",
                "import Foundation\n#if canImport(FoundationNetworking)\nimport FoundationNetworking\n#endif",
            )
            print(f"Added FoundationNetworking to {swift_file.name}")

        guard_stmt = None
        if "import CoreGraphics" in content or "import Vision" in content:
            guard_stmt = "#if canImport(CoreGraphics)"
        elif "import CoreML" in content or "CoreML" in swift_file.parts:
            guard_stmt = "#if canImport(CoreML)"

        if guard_stmt and not content.startswith(guard_stmt):
            content = f"{guard_stmt}\n" + content + "\n#endif\n"
            print(f"Wrapped {swift_file.name} in {guard_stmt} guard")

        if content != original_content:
            swift_file.write_text(content)

    ccv_dir = SOURCE / ".build" / "checkouts" / "ccv"

    # Patch: Ensure PAGE_SIZE is defined in ccv_nnc_easy.h on Linux platforms
    easy_h = ccv_dir / "lib" / "nnc" / "ccv_nnc_easy.h"
    if easy_h.is_file():
        print("Patching ccv_nnc_easy.h with PAGE_SIZE fallback...")
        content = easy_h.read_text()
        if "#ifndef PAGE_SIZE" not in content:
            fallback = "\n#ifndef PAGE_SIZE\n#define PAGE_SIZE 4096\n#endif\n"
            easy_h.write_text(fallback + content)

    # Patch: Guard 'dispatch/dispatch.h' inclusion with #if defined(__APPLE__) in all C/C++ files inside ccv
    print("Wrapping <dispatch/dispatch.h> and <Accelerate/Accelerate.h> references with platform guards...")
    for filepath in ccv_dir.glob("**/*"):
        if filepath.is_file() and filepath.suffix in (".c", ".h", ".cpp", ".hpp", ".m"):
            content = filepath.read_text()
            changed = False
            if "#include <dispatch/dispatch.h>" in content:
                content = content.replace(
                    "#include <dispatch/dispatch.h>", "#if defined(__APPLE__)\n#include <dispatch/dispatch.h>\n#endif"
                )
                changed = True
            if "#include <Accelerate/Accelerate.h>" in content:
                content = content.replace(
                    "#include <Accelerate/Accelerate.h>",
                    "#if defined(__APPLE__)\n#include <Accelerate/Accelerate.h>\n#else\n#include <cblas.h>\n#endif",
                )
                changed = True
            if changed:
                filepath.write_text(content)

    print("Recursively wrapping all MPS/MFA subdirectories and XPU files with Linux bypass preprocessors...")
    for filepath in ccv_dir.glob("**/*"):
        if filepath.is_file() and filepath.suffix in (".m", ".h", ".c", ".cpp", ".hpp"):
            parts = filepath.parts
            if "mps" in parts or "mfa" in parts:
                content = filepath.read_text()
                if not content.strip().startswith("#if defined(__APPLE__)"):
                    patched_content = "#if defined(__APPLE__)\n" + content + "\n#endif\n"
                    filepath.write_text(patched_content)
            elif filepath.name == "ccv_nnc_xpu_alloc.c":
                content = filepath.read_text()
                if not content.strip().startswith("#if defined(__APPLE__)"):
                    patched_content = (
                        "#if defined(__APPLE__)\n"
                        + content
                        + "\n#else\n#include <stdlib.h>\nvoid* ccv_nnc_xpu_alloc(void* a, const int b, void* c, const size_t d) { return malloc(d); }\nvoid ccv_nnc_xpu_free(void* a, void* b) { free(b); }\nvoid ccv_nnc_xpu_alloc_destroy(void* a) {}\nvoid ccv_nnc_xpu_gc(const int a, void* b) {}\n#endif\n"
                    )
                    filepath.write_text(patched_content)

            # Apply mpobjmalloc/mpobjfree fallback for any files that implicitly call it
            content = filepath.read_text()
            if (
                ("mpobjmalloc" in content or "mpobjfree" in content)
                and "#define mpobjmalloc" not in content
                and filepath.name != "ccv_nnc_xpu_alloc.c"
            ):
                filepath.write_text(
                    "#if !defined(__APPLE__)\n#include <stdlib.h>\n#define mpobjmalloc(d, s) malloc(s)\n#define mpobjfree(d, p) free(p)\n#endif\n"
                    + content
                )

            # Apply mpmemcpy fallback for any files that implicitly call it
            content = filepath.read_text()
            if "mpmemcpy" in content and "#define mpmemcpy" not in content and filepath.name != "ccv_nnc_xpu_alloc.c":
                filepath.write_text(
                    "#if !defined(__APPLE__)\n#include <string.h>\n#include <stdint.h>\n#define mpmemcpy(dest, dest_off, dest_type, src, src_off, src_type, size) memcpy((uint8_t*)(dest) + (dest_off), (const uint8_t*)(src) + (src_off), (size))\n#endif\n"
                    + content
                )

            # Apply mpheapalloc/mpheapfree/mpobjcreate fallback for any files that implicitly call it
            content = filepath.read_text()
            if (
                ("mpheapalloc" in content or "mpobjcreate" in content)
                and "#define mpheapalloc" not in content
                and filepath.name != "ccv_nnc_xpu_alloc.c"
            ):
                filepath.write_text(
                    "#if !defined(__APPLE__)\n#include <stdlib.h>\n#include <stdint.h>\n#define mpheapalloc(d, s) malloc(s)\n#define mpheapfree(d, p) free(p)\n#define mpobjcreate(p, o, s) ((void*)((uint8_t*)(p) + (o)))\n#endif\n"
                    + content
                )

            # Fix dispatch_async_f for Linux in ccv_nnc_stream.c and others
            content = filepath.read_text()
            if "dispatch_async_f" in content and "#define dispatch_async_f" not in content:
                filepath.write_text(
                    "#if !defined(__APPLE__)\n#define DISPATCH_QUEUE_PRIORITY_DEFAULT 0\ntypedef void (*dispatch_function_t)(void *);\n#define dispatch_get_global_queue(id, flags) 0\n#define dispatch_async_f(queue, context, work) ((dispatch_function_t)(work))(context)\n#endif\n"
                    + content
                )

            # Fix HAVE_MPS checks so they require Apple platforms to be defined
            content = filepath.read_text()
            new_content = content.replace("defined(HAVE_MPS)", "(defined(HAVE_MPS) && defined(__APPLE__))")
            new_content = new_content.replace("#ifdef HAVE_MPS", "#if defined(HAVE_MPS) && defined(__APPLE__)")
            if new_content != content:
                filepath.write_text(new_content)

            # Stub ccv_nnc_mps functions for Linux just in case
            content = filepath.read_text()
            stubs = ""
            if (
                "ccv_nnc_mps_clear_graph_executable_cache" in content
                and "#define ccv_nnc_mps_clear_graph_executable_cache" not in content
            ):
                stubs += "#if !defined(__APPLE__)\n#define ccv_nnc_mps_clear_graph_executable_cache()\n#endif\n"
            if (
                "ccv_nnc_mps_set_queue_watermark" in content
                and "#define ccv_nnc_mps_set_queue_watermark" not in content
            ):
                stubs += "#if !defined(__APPLE__)\n#define ccv_nnc_mps_set_queue_watermark(w)\n#endif\n"
            if (
                "ccv_nnc_mps_set_binary_artifacts" in content
                and "#define ccv_nnc_mps_set_binary_artifacts" not in content
            ):
                stubs += "#if !defined(__APPLE__)\n#define ccv_nnc_mps_set_binary_artifacts(a, b, c)\n#endif\n"
            if stubs:
                filepath.write_text(stubs + content)

    os.environ["PKG_CONFIG_PATH"] = "/usr/lib/x86_64-linux-gnu/pkgconfig:" + os.environ.get("PKG_CONFIG_PATH", "")

    jobs = os.cpu_count() or 2
    # Built with explicit module map search paths to resolve CZlib issues seamlessly
    sh(
        f"swift build -c release --product {CLI_NAME} -j {jobs} -Xcc -I/usr/include -Xcc -I/tmp/CZlib -Xswiftc -I/tmp/CZlib -Xcc -I/tmp/SQLite3 -Xswiftc -I/tmp/SQLite3 -Xlinker -L/usr/lib -Xcc -DCCV_DISABLE_MPS -Xswiftc -DCCV_DISABLE_MPS",
        cwd=SOURCE,
    )
    if use_drive_cache and BINARY.is_file():
        DRIVE_CACHE.mkdir(parents=True, exist_ok=True)
        shutil.copy2(BINARY, CACHED_BINARY)
        print("saved the converter to Drive:", CACHED_BINARY)

if not BINARY.is_file():
    raise RuntimeError("The build finished without producing the converter binary.")

draw_things_revision = subprocess.run(
    ["git", "-C", str(SOURCE), "rev-parse", "HEAD"], capture_output=True, text=True
).stdout.strip()
print("converter ready:", BINARY)
print("media-generation-kit revision:", PINNED_REVISION)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
$ git -C /content/mgk-build/media-generation-kit checkout 37e0b70092ad1a0c1d4c6b24f16e17b73f1b1fb3
HEAD is now at 37e0b70 Ping by revision.
$ swift package resolve
ZIPFoundation Linux patch ready
Added FoundationNetworking to ConfigurationZoo.swift
Added FoundationNetworking to MediaGenerationResourceLoader.swift
Added FoundationNetworking to CloudAuthenticator.swift
Added FoundationNetworking to LoRAStore.swift
Added FoundationNetworking to MediaGenerationCloudAuthentication.swift
Added FoundationNetworking to ResumableDownloader.swift
Added FoundationNetworking to TextualInversionZoo+Downloads.swift
Added FoundationNetworking to ModelZoo+Downloads.swift
Added FoundationNetworking to LoRAZoo+Downloads.swift
Added FoundationNetworking to ControlNetZoo+Downloads.swift
Patching ccv_nnc_easy.h with PAGE_SIZE fallback...
Wrapping <dispatch/dispatch.h> and <Accele

In [11]:
import subprocess
from pathlib import Path

WORK = Path("/content/mgk-build")
SOURCE = WORK / "media-generation-kit"
xpu_alloc = SOURCE / ".build" / "checkouts" / "ccv" / "lib" / "nnc" / "ccv_nnc_xpu_alloc.c"

# Let's check the original file content by unstubbing it temporarily or reading it directly if we can fetch it via git
print("--- Inspecting ccv_nnc_xpu_alloc.c ---")
subprocess.run(f"git checkout {xpu_alloc}", shell=True, cwd=SOURCE)
content = xpu_alloc.read_text()

# Extract and print anything that looks like a function definition
lines = content.split("\n")
for i, line in enumerate(lines):
    if "mpobjmalloc" in line or "mpobjfree" in line or "xpu_alloc" in line:
        print(f"Line {i + 1}: {line}")

--- Inspecting ccv_nnc_xpu_alloc.c ---
Line 7: #include "_ccv_nnc_xpu_alloc.h"
Line 30: 		mpobjfree(node->device, node->ptr);
Line 37: static void _ccv_nnc_xpu_alloc_drain(const int device, khash_t(dy_dev)* const dev, const ccv_nnc_stream_context_t* const stream)
Line 66: 	ccv_nnc_xpu_alloc_t* const xpu_alloc = (ccv_nnc_xpu_alloc_t*)context;
Line 67: 	khash_t(dy_str)* const freed = xpu_alloc->freed;
Line 72: 	_ccv_nnc_xpu_alloc_drain(-1, dev, stream);
Line 77: void* ccv_nnc_xpu_alloc(ccv_nnc_xpu_alloc_t* const xpu_alloc, const int device, ccv_nnc_stream_context_t* const stream, const size_t size)
Line 79: 	khash_t(dy_str)* const freed = xpu_alloc->freed;
Line 113: 		kh_val(freed, i).hook_id = stream ? ccv_nnc_stream_context_add_destructor_hook(stream, _ccv_nnc_xpu_stream_destructor_hook, xpu_alloc) : -1;
Line 120: 		if (xpu_alloc->mp_hdr < 0)
Line 121: 			xpu_alloc->mp_hdr = curegmp(device, (cump_f)ccv_nnc_xpu_gc, xpu_alloc);
Line 124: 		if (xpu_alloc->mp_hdr < 0)
Line 125: 			xpu_allo

In [ ]:
# @title 3. Convert a LoRA (pick the file, then download the .ckpt)
# @markdown Scale is applied to the LoRA weights while converting. Leave 1.0 unless you know you need another value.
scale = 1.0  # @param {type:"number"}
# @markdown Leave empty to pick a file from your computer. Or give a path inside Colab / Drive, e.g. /content/drive/MyDrive/my_lora.safetensors
source_path = ""  # @param {type:"string"}
download_result = True  # @param {type:"boolean"}

if source_path.strip():
    source = Path(source_path.strip()).expanduser()
else:
    from google.colab import files

    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError("Pick exactly one .safetensors file.")
    source = Path("/content") / next(iter(uploaded))

if not source.is_file():
    raise RuntimeError(f"Input not found: {source}")
if source.suffix.lower() != ".safetensors":
    print(f"warning: {source.name} does not end in .safetensors")


def sha256_of(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        while chunk := handle.read(1024 * 1024):
            digest.update(chunk)
    return digest.hexdigest()


out_dir = Path("/content/lora_out")
out_dir.mkdir(exist_ok=True)
with tempfile.TemporaryDirectory(prefix=".convert-", dir=out_dir) as staging:
    command = [str(BINARY), "lora", "convert", "--input", str(source), "--output-dir", staging]
    if scale != 1.0:
        command += ["--scale", str(scale)]
    sh(shlex.join(command))
    produced = list(Path(staging).glob("*.ckpt"))
    if len(produced) != 1:
        raise RuntimeError(f"Expected exactly one .ckpt from the converter, found {len(produced)}.")
    target = out_dir / produced[0].name
    if target.exists():
        raise RuntimeError(f"{target} already exists; not overwriting it. Remove it first.")
    produced[0].replace(target)

size = target.stat().st_size
digest = sha256_of(target)
provenance = target.with_suffix(".json")
provenance.write_text(
    json.dumps(
        {
            "output": target.name,
            "size_bytes": size,
            "sha256": digest,
            "input": source.name,
            "input_sha256": sha256_of(source),
            "scale": scale,
            "media_generation_kit_revision": PINNED_REVISION,
            "swift_version": SWIFT_VERSION,
        },
        indent=2,
    )
)
print("output:", target)
print("size:  ", size, "bytes")
print("sha256:", digest)

if download_result:
    from google.colab import files

    files.download(str(target))
    files.download(str(provenance))